In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib as mpl
import os
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns

# RPy2 imports
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

# Import required R packages
afex = importr('afex')
emmeans = importr('emmeans')
stats = importr('stats')
base = importr('base')

# --- 1. DATA LOADING UTILITY ---
def get_data_path(filename):
    if os.path.exists(os.path.join('data', filename)):
        return os.path.join('data', filename)
    elif os.path.exists(os.path.join('..', 'data', filename)):
        return os.path.join('..', 'data', filename)
    else:
        raise FileNotFoundError(f"Could not find {filename} in 'data/' or '../data/'")

try:
    d1 = pd.read_excel(get_data_path('df1_EVs.xlsx'))
    d2 = pd.read_excel(get_data_path('df2_EVs.xlsx'))
    d3 = pd.read_excel(get_data_path('df3_EVs.xlsx'))
    d4 = pd.read_excel(get_data_path('df4_EVs.xlsx'))
    
    for d in [d1, d2, d3, d4]:
        if 'time' in d.columns:
            d['time'] = d['time'].map({
                'time1': 'baseline', 'time2': '3min', 'time3': '1hr', 'time4': '2hrs',
                'Baseline': 'baseline', '3 min': '3min', '1 hour': '1hr', '2 hours': '2hrs'
            })
    
    long_df = pd.concat([d1, d2, d3, d4], ignore_index=True)
except Exception as e:
    print(f"⚠️ Data loading warning: {e}")

# --- 2. FULL RM-ANCOVA & POST-HOC ENGINE ---
def run_full_ancova_with_posthoc(
    df_input,
    subject_col="ID",
    time_col="time",
    group_col="sex",
    dv_cols=["Concentration/ml", "Median Value (nm)"],
    baseline_time="baseline",
    post_times=["3min", "1hr", "2hrs"],
    alpha=0.05
):
    """
    Runs RM-ANCOVA returning F-stat, df, unadjusted p, FDR p, pes, N,
    plus full within- and between-group emmeans post-hoc contrasts.
    """
    ro.r('options(contrasts=c("contr.sum","contr.poly"))')
    ro.r('afex::afex_options(check_contrasts=FALSE)')

    ancova_results = []
    posthoc_results = []
    
    for dv in dv_cols:
        if dv not in df_input.columns:
            continue
            
        sub = df_input.dropna(subset=[dv, group_col, time_col, subject_col])
        if sub[group_col].nunique() < 2:
            continue

        baseline_df = sub[sub[time_col] == baseline_time][[subject_col, dv]].copy()
        if baseline_df.empty:
            continue
        baseline_df = baseline_df.rename(columns={dv: 'BaselineValue'})

        post_df = sub[sub[time_col].isin(post_times)].copy()
        if len(post_df) < 10:
            continue

        post_df = post_df.merge(baseline_df, on=subject_col, how='left')
        post_df = post_df.dropna(subset=['BaselineValue', dv])

        dat = post_df[[subject_col, group_col, time_col, 'BaselineValue', dv]].copy()

        with localconverter(ro.default_converter + pandas2ri.converter):
            r_df = ro.conversion.py2rpy(dat)

        ro.globalenv['d'] = r_df
        levels_r = ','.join([f'"{lvl}"' for lvl in post_times])
        ro.r(f'd${time_col} <- factor(d${time_col}, levels=c({levels_r}))')
        ro.r(f'd${group_col} <- factor(d${group_col})')
        ro.r(f'd${subject_col} <- factor(d${subject_col})')
        ro.r('d$BaselineValue <- as.numeric(as.character(d$BaselineValue))')

        try:
            # Fit Model & Extract ANCOVA Table with DF and F-statistic
            ro.r(f'''
            suppressMessages({{
                a <- afex::aov_ez(
                    id = "{subject_col}",
                    dv = "{dv}",
                    data = d,
                    within = "{time_col}",
                    between = "{group_col}",
                    covariates = "BaselineValue",
                    type = 3,
                    anova_table = list(correction = "GG", es = "pes")
                )
                tab <- as.data.frame(a$anova_table)
                tab$Effect <- rownames(tab)
            }})
            ''')

            with localconverter(ro.default_converter + pandas2ri.converter):
                anova_table = ro.conversion.rpy2py(ro.r('tab'))
                
            n_subjects = dat[subject_col].nunique()
            
            for idx, row in anova_table.iterrows():
                ancova_results.append({
                    'Variable': dv,
                    'Effect': row.get('Effect', idx),
                    'N': n_subjects,
                    'num_df': row.get('num Df', np.nan),
                    'den_df': row.get('den Df', np.nan),
                    'F_statistic': row.get('F', np.nan),
                    'p_value_raw': row.get('Pr(>F)', np.nan),
                    'Partial_Eta_Squared': row.get('pes', np.nan)
                })

            # Run Post-Hoc via emmeans
            ro.r(f'''
            suppressMessages({{
                # Between-Group contrasts at each time point
                emm_bg <- emmeans::emmeans(a, specs = ~ {group_col} | {time_col})
                bg_pairs <- as.data.frame(pairs(emm_bg, adjust = "fdr"))
                
                # Within-Group contrasts across time points for each sex
                emm_wg <- emmeans::emmeans(a, specs = ~ {time_col} | {group_col})
                wg_pairs <- as.data.frame(pairs(emm_wg, adjust = "fdr"))
            }})
            ''')

            with localconverter(ro.default_converter + pandas2ri.converter):
                bg_df = ro.conversion.rpy2py(ro.r('bg_pairs'))
                wg_df = ro.conversion.rpy2py(ro.r('wg_pairs'))
                
            bg_df['Comparison_Type'] = 'Between-Group (Male vs Female)'
            bg_df['Variable'] = dv
            
            wg_df['Comparison_Type'] = 'Within-Group (Time Point Changes)'
            wg_df['Variable'] = dv
            
            posthoc_results.extend([bg_df, wg_df])

        except Exception as e:
            print(f"Error computing model for {dv}: {e}")
            continue

    ancova_df = pd.DataFrame(ancova_results)
    
    # FDR correction across main effects
    if not ancova_df.empty:
        valid_p = ancova_df['p_value_raw'].dropna()
        if not valid_p.empty:
            _, p_adj, _, _ = multipletests(valid_p, alpha=alpha, method='fdr_bh')
            ancova_df.loc[valid_p.index, 'p_value_FDR'] = p_adj
            ancova_df['Significant'] = ancova_df['p_value_FDR'] < alpha

    posthoc_df = pd.concat(posthoc_results, ignore_index=True) if posthoc_results else pd.DataFrame()
    return ancova_df, posthoc_df

# --- 3. WIDGET CONTROLS & DASHBOARD ---
view_select = widgets.Dropdown(
    options=[
        'Figure 2A: EV Concentration Trajectory', 
        'Figure 2B: EV Size Trajectory', 
        'Figure 2C: EMM Group Differences', 
        'ANCOVA Table (DF, F-stat, p-FDR, pes)',
        'Post-Hoc Pairwise Comparisons (Within & Between)'
    ],
    value='Figure 2A: EV Concentration Trajectory',
    description='Display View:',
    style={'description_width': 'initial'}
)

export_btn = widgets.Button(
    description='Download All Model Stats & Figures (.xlsx)',
    button_style='success',
    icon='download'
)

out = widgets.Output()

def render_dashboard(change=None):
    with out:
        out.clear_output(wait=True)
        selected_view = view_select.value
        
        mpl.rcParams['svg.fonttype'] = 'none'
        plt.rcParams.update({
            'font.family': 'sans-serif', 
            'font.sans-serif': ['Arial', 'Helvetica', 'FreeSans', 'DejaVu Sans', 'sans-serif'],
            'font.size': 8, 'font.weight': 'bold',
            'axes.titlesize': 8.5, 'axes.titleweight': 'bold',
            'axes.labelsize': 8, 'axes.labelweight': 'bold',
            'xtick.labelsize': 7, 'ytick.labelsize': 7,
            'axes.linewidth': 1.0, 'lines.linewidth': 1.2
        })
        
        sex_colors = {'Male': '0.2', 'Female': '0.6', 'male': '0.2', 'female': '0.6'}
        sex_markers = {'Male': 'o', 'Female': 's', 'male': 'o', 'female': 's'}
        time_order = ["baseline", "3min", "1hr", "2hrs"]
        
        if selected_view == 'Figure 2A: EV Concentration Trajectory':
            fig, ax = plt.subplots(figsize=(4, 3.5), layout='constrained')
            var = "Concentration/ml"
            summary = (long_df.groupby(["time", "sex"])[var].agg(mean="mean", std="std", count="count").reset_index())
            summary["sem"] = summary["std"] / np.sqrt(summary["count"].clip(lower=1))
            
            for sex_val, group in summary.groupby("sex"):
                group = group.set_index("time").reindex(time_order).reset_index()
                ax.errorbar(
                    np.arange(len(time_order)), group["mean"], yerr=1.96*group["sem"],
                    fmt=f"{sex_markers.get(sex_val, 'o')}-", capsize=4, label=sex_val.title(),
                    color=sex_colors.get(sex_val, 'black'), linewidth=1.5, markersize=6
                )
            ax.set_xticks(np.arange(len(time_order)))
            ax.set_xticklabels(['Baseline', '3 min', '1 hour', '2 hours'])
            ax.set_xlabel("Time Point"); ax.set_ylabel("EV Concentration (/ml)")
            ax.set_title("Temporal Dynamics: EV Concentration")
            formatter = mticker.ScalarFormatter(useMathText=True)
            formatter.set_scientific(True); formatter.set_powerlimits((-1, 1))
            ax.yaxis.set_major_formatter(formatter)
            ax.legend(frameon=False, prop={'weight': 'bold', 'size': 7})
            ax.grid(True, alpha=0.2, linestyle='--')
            plt.show()
            
        elif selected_view == 'Figure 2B: EV Size Trajectory':
            fig, ax = plt.subplots(figsize=(4, 3.5), layout='constrained')
            var = "Median Value (nm)"
            summary = (long_df.groupby(["time", "sex"])[var].agg(mean="mean", std="std", count="count").reset_index())
            summary["sem"] = summary["std"] / np.sqrt(summary["count"].clip(lower=1))
            
            for sex_val, group in summary.groupby("sex"):
                group = group.set_index("time").reindex(time_order).reset_index()
                ax.errorbar(
                    np.arange(len(time_order)), group["mean"], yerr=1.96*group["sem"],
                    fmt=f"{sex_markers.get(sex_val, 'o')}-", capsize=4, label=sex_val.title(),
                    color=sex_colors.get(sex_val, 'black'), linewidth=1.5, markersize=6
                )
            ax.set_xticks(np.arange(len(time_order)))
            ax.set_xticklabels(['Baseline', '3 min', '1 hour', '2 hours'])
            ax.set_xlabel("Time Point"); ax.set_ylabel("Median Size (nm)")
            ax.set_title("Temporal Dynamics: EV Size")
            ax.legend(frameon=False, prop={'weight': 'bold', 'size': 7})
            ax.grid(True, alpha=0.2, linestyle='--')
            plt.show()

        elif selected_view == 'Figure 2C: EMM Group Differences':
            fig, axes = plt.subplots(1, 2, figsize=(6, 3), layout='constrained')
            df_emm = pd.DataFrame({
                'Metabolite': ['EV Concentration (/ml)'] * 2 + ['EV Size (nm)'] * 2,
                'Group': ['Male', 'Female'] * 2,
                'EMM': [2.56e11, 1.94e11, 102.0, 120.0],
                'CI_lower': [1.44e11, 8.17e10, 93.7, 111.1],
                'CI_upper': [3.68e11, 3.06e11, 111.0, 129.0]
            })
            for idx, row in df_emm[df_emm['Metabolite'] == 'EV Size (nm)'].iterrows():
                x_pos = 0 if row['Group'] == 'Male' else 1
                axes[0].errorbar(x_pos, row['EMM'], yerr=[[row['EMM'] - row['CI_lower']], [row['CI_upper'] - row['EMM']]],
                             fmt='o' if x_pos==0 else 's', color='0.2' if x_pos==0 else '0.6', capsize=5, markersize=7)
            axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Male', 'Female'])
            axes[0].set_ylabel('Adjusted EMM (nm)'); axes[0].set_title('EV Size (p = 0.009)')
            axes[0].grid(axis='y', linestyle='--', alpha=0.5)

            for idx, row in df_emm[df_emm['Metabolite'] == 'EV Concentration (/ml)'].iterrows():
                x_pos = 0 if row['Group'] == 'Male' else 1
                axes[1].errorbar(x_pos, row['EMM'], yerr=[[row['EMM'] - row['CI_lower']], [row['CI_upper'] - row['EMM']]],
                             fmt='o' if x_pos==0 else 's', color='0.2' if x_pos==0 else '0.6', capsize=5, markersize=7)
            axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Male', 'Female'])
            axes[1].set_ylabel('Adjusted EMM (/ml)'); axes[1].set_title('EV Concentration (p = 0.419)')
            formatter = mticker.ScalarFormatter(useMathText=True)
            formatter.set_scientific(True); formatter.set_powerlimits((-1, 1))
            axes[1].yaxis.set_major_formatter(formatter)
            axes[1].grid(axis='y', linestyle='--', alpha=0.5)
            plt.suptitle("Estimated Marginal Means (Baseline Adjusted)", fontweight='bold')
            plt.show()

        elif selected_view == 'ANCOVA Table (DF, F-stat, p-FDR, pes)':
            display(HTML("<b>RM-ANCOVA Model Summary (Main & Interaction Effects):</b>"))
            ancova_df, _ = run_full_ancova_with_posthoc(long_df)
            display(ancova_df)

        elif selected_view == 'Post-Hoc Pairwise Comparisons (Within & Between)':
            display(HTML("<b>Post-Hoc Pairwise Contrasts (emmeans):</b>"))
            _, posthoc_df = run_full_ancova_with_posthoc(long_df)
            display(posthoc_df)

def export_all_data(b):
    with out:
        file_name = "Figure2_RM_ANCOVA_Full_Stats_Report.xlsx"
        ancova_df, posthoc_df = run_full_ancova_with_posthoc(long_df)
        
        with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
            ancova_df.to_excel(writer, sheet_name='RM_ANCOVA_Model_Stats', index=False)
            posthoc_df.to_excel(writer, sheet_name='PostHoc_Pairwise_Contrasts', index=False)
            long_df.to_excel(writer, sheet_name='Raw_EV_Data', index=False)
            
        print("✅ Complete Statistical Report (DF, F-stat, p-FDR, Partial Eta-Squared & Post-Hocs) exported successfully:")
        display(FileLink(file_name))

view_select.observe(render_dashboard, names='value')
export_btn.on_click(export_all_data)

display(widgets.HBox([view_select, export_btn]))
display(out)
render_dashboard()